In [ ]:
import sys, os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "Global-Financial-Firm-1B-kickstarter-crowdfunding-recommendation-engine"
    GITHUB_URL = f"https://github.com/Break-Through-Tech/{REPO}.git"
    if not os.path.exists(REPO):
        os.system(f"git clone -q {GITHUB_URL}")
    os.chdir(f"{REPO}/notebooks")

sys.path.insert(0, "..")
print("Colab" if IN_COLAB else "local", "| cwd:", Path.cwd().name)

Colab | cwd: notebooks


In [ ]:
"""Milestone 3: Logistic Regression for Kickstarter success prediction.

Run from the repository root:

    python logistic_regression_m3.py

What it does
    1. Rebuilds the modelling table from the raw zip (load_raw -> clean -> build_features)
    2. For each split (random, chronological) fits the stateful features on TRAIN only
    3. For each feature set (launch_only, plus_campaign) fits:
         - the default Logistic Regression from make_models()
         - a tuned Logistic Regression (GridSearchCV on C and L1/L2)
    4. Scores everything through evaluate() so rows match the other models
    5. Writes tables and figures to reports/

launch_only is the honest, deployable model. plus_campaign (adds `updates`)
exists only to compare against the published 80.2% benchmark.
"""
from __future__ import annotations

from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # save figures without needing a display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold, TimeSeriesSplit

from src.cleaning import clean
from src.evaluation import (
    comparison_table,
    confusion_frame,
    describe_errors,
    evaluate,
    majority_class_baseline,
    roc_points,
)
from src.features import (
    add_location_frequency,
    add_relative_goal,
    build_features,
    build_model_matrix,
)
from src.loading import load_raw
from src.modeling import (
    RANDOM_STATE,
    align_columns,
    chronological_split,
    make_models,
    random_split,
    split_summary,
)

REPORTS = Path("reports")
REPORTS.mkdir(exist_ok=True)

SPLITS = {"random": random_split, "chronological": chronological_split}

# Stateful features are added after the split (see features.py), then passed in
# through extra_numeric. `log_updates` only joins the benchmark feature set.
EXTRA_LAUNCH = ["log_city_project_count", "log_goal_vs_subcat_median"]
FEATURE_SETS = {
    "launch_only": EXTRA_LAUNCH,
    "plus_campaign": EXTRA_LAUNCH + ["log_updates"],
}

# Optimized parameters to speed up execution time
PARAM_GRID = {
    "clf__C": [0.1, 1.0],
    "clf__penalty": ["l1", "l2"],
    "clf__solver": ["liblinear"],  # much faster than 'saga' for this size
    "clf__max_iter": [1000],
}

CATEGORICAL_PREFIXES = ("category_", "subcategory_", "state_raw_")


# ---------------------------------------------------------------------------
# Data preparation
# ---------------------------------------------------------------------------
def prepare_split(df: pd.DataFrame, splitter):
    """Split, then fit city-frequency and subcategory-median on train only."""
    train, test = splitter(df)

    train, city_ref = add_location_frequency(train)
    test, _ = add_location_frequency(test, city_ref)

    train, goal_ref = add_relative_goal(train)
    test, _ = add_relative_goal(test, goal_ref)

    for part in (train, test):
        part["log_updates"] = np.log1p(part["updates"])
    return train, test


def make_xy(train, test, feature_set: str, extra: list[str]):
    """Build matrices, align columns, and impute with TRAIN medians."""
    X_train, y_train = build_model_matrix(train, feature_set, extra_numeric=extra)
    X_test, y_test = build_model_matrix(test, feature_set, extra_numeric=extra)
    X_test = align_columns(X_train, X_test)

    # Guard against inf (e.g. a goal of 0 in a ratio feature) and any NaN
    # introduced by alignment; fill from training data only.
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    medians = X_train.median()
    return X_train.fillna(medians), y_train, X_test.fillna(medians), y_test


# ---------------------------------------------------------------------------
# Modelling
# ---------------------------------------------------------------------------
def tune(pipe, X_train, y_train, chronological: bool) -> GridSearchCV:
    """Grid-search C / penalty. Chronological runs use time-ordered CV so a
    validation fold never comes before its training data."""
    # Reduced splits from 5 to 3 for significantly faster tuning runs
    cv = (
        TimeSeriesSplit(n_splits=3)
        if chronological
        else StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    )
    gs = GridSearchCV(clone(pipe), PARAM_GRID, scoring="roc_auc", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    return gs


# ---------------------------------------------------------------------------
# Interpretation
# ---------------------------------------------------------------------------
def coefficient_table(model, columns) -> pd.DataFrame:
    """Coefficients on two scales.

    per_sd   : effect of a +1 standard-deviation change. Comparable across
               NUMERIC features, so use it to rank them.
    per_unit : effect in the feature's original units. For one-hot columns
               this is 'being in this category vs not', so use it for the
               category / subcategory / state effects.
    """
    scale = model.named_steps["scale"].scale_
    coef_sd = model.named_steps["clf"].coef_[0]
    coef_unit = coef_sd / scale

    out = pd.DataFrame(
        {
            "feature": columns,
            "kind": np.where(
                pd.Index(columns).str.startswith(CATEGORICAL_PREFIXES),
                "categorical",
                "numeric",
            ),
            "coef_per_sd": coef_sd,
            "coef_per_unit": coef_unit,
            "odds_ratio_per_sd": np.exp(coef_sd),
            "odds_ratio_per_unit": np.exp(coef_unit),
        }
    )
    out["pct_odds_change_per_sd"] = ((out["odds_ratio_per_sd"] - 1) * 100).round(1)
    out["pct_odds_change_per_unit"] = ((out["odds_ratio_per_unit"] - 1) * 100).round(1)
    return out


def plot_bars(df: pd.DataFrame, value: str, title: str, xlabel: str, path: Path, top: int = 10):
    df = df[df[value] != 0]
    pos = df[df[value] > 0].nlargest(top, value)
    neg = df[df[value] < 0].nsmallest(top, value)
    plot_df = pd.concat([pos, neg]).sort_values(value)
    if plot_df.empty:
        return
    colors = np.where(plot_df[value] > 0, "#2a9d8f", "#e76f51")
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(plot_df) + 1.5))
    ax.barh(plot_df["feature"], plot_df[value], color=colors)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_roc(curves: dict, title: str, path: Path):
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for name, (fpr, tpr, auc) in curves.items():
        ax.plot(fpr, tpr, label=f"{name} (AUC {auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Chance")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    raw = load_raw("../data/DSI_kickstarterscrape_dataset_csv.zip")
    clean_df, _ = clean(raw, verbose=False)
    df = build_features(clean_df)
    print(f"Modelling table: {len(df):,} rows\n")

    all_rows = []

    for split_name, splitter in SPLITS.items():
        print("=" * 72)
        train, test = prepare_split(df, splitter)
        print(split_summary(train, test, split_name).to_string(index=False))

        roc_curves = {}
        baseline_done = False

        for fs_name, extra in FEATURE_SETS.items():
            tag = f"{split_name}/{fs_name}"
            print(f"\n--- {tag} ---")
            X_train, y_train, X_test, y_test = make_xy(train, test, fs_name, extra)

            if not baseline_done:
                all_rows.append(majority_class_baseline(y_train, y_test).assign(split=split_name))
                baseline_done = True

            base_pipe = make_models()["Logistic Regression"]
            base = clone(base_pipe).fit(X_train, y_train)

            gs = tune(base_pipe, X_train, y_train, chronological=(split_name == "chronological"))
            tuned = gs.best_estimator_
            print(f"best params: {gs.best_params_}  (CV AUC {gs.best_score_:.4f})")

            all_rows.append(evaluate(base, X_test, y_test, label="Logistic Regression",
                                     feature_set=fs_name, split=split_name))
            all_rows.append(evaluate(tuned, X_test, y_test, label="Logistic Regression (tuned)",
                                     feature_set=fs_name, split=split_name))

            print("\nConfusion matrix (tuned)")
            print(confusion_frame(tuned, X_test, y_test))

            errors = describe_errors(tuned, X_test, y_test, test[["goal", "duration_days"]])
            print("\nError breakdown (tuned)")
            print(errors)
            errors.to_csv(REPORTS / f"m3_lr_errors_{split_name}_{fs_name}.csv")

            coefs = coefficient_table(tuned, X_train.columns)
            coefs.to_csv(REPORTS / f"m3_lr_coefficients_{split_name}_{fs_name}.csv", index=False)

            numeric = coefs[coefs["kind"] == "numeric"]
            categorical = coefs[coefs["kind"] == "categorical"]
            plot_bars(numeric, "coef_per_sd",
                      f"Numeric drivers ({tag})",
                      "Log-odds change per +1 SD", REPORTS / f"m3_lr_numeric_{split_name}_{fs_name}.png")
            plot_bars(categorical, "coef_per_unit",
                      f"Category / state effects ({tag})",
                      "Log-odds change vs. not in category", REPORTS / f"m3_lr_categorical_{split_name}_{fs_name}.png")

            top = numeric.reindex(numeric["coef_per_sd"].abs().sort_values(ascending=False).index).head(8)
            print("\nTop numeric drivers")
            print(top[["feature", "pct_odds_change_per_sd"]].to_string(index=False))

            roc_curves[fs_name] = roc_points(tuned, X_test, y_test)

        plot_roc(roc_curves, f"Logistic Regression ROC ({split_name} split)",
                 REPORTS / f"m3_lr_roc_{split_name}.png")

    table = comparison_table(*all_rows, sort_by="roc_auc")
    table.to_csv(REPORTS / "m3_lr_results.csv", index=False)
    print("\n" + "=" * 72)
    print(table.round(4).to_string(index=False))


if __name__ == "__main__":
    main()

Modelling table: 38,378 rows

 split  train_n  test_n  train_success_rate  test_success_rate  train_end test_start
random    30702    7676              0.5474             0.5473 2012-05-29 2009-04-28

--- random/launch_only ---
best params: {'clf__C': 0.1, 'clf__max_iter': 1000, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}  (CV AUC 0.7689)

Confusion matrix (tuned)
                    predicted: failed  predicted: successful
actual: failed                   2164                   1311
actual: successful                866                   3335

Error breakdown (tuned)
                                                n  median_goal  \
error_type                                                       
true positive                                3335       2500.0   
true negative                                2164       8700.0   
false positive (told to launch, would fail)  1311       3000.0   
false negative (discouraged, would succeed)   866       5500.0   

                      